# Gold dimension -- `dbo.dim_product`

Conformed product dimension with price history

**Source:** `silver.stg_products`  
**SCD:** `type2`  
**Surrogate key:** `product_sk`  
**Tracked columns:** `list_price, category`

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
# Reads from lh_silver, writes to wh_gold. Both must be
# attached to this notebook; wh_gold must be the DEFAULT so an
# unqualified write cannot land in the wrong item.
target_item = "wh_gold"
source_item = "lh_silver"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="dim_product",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="gold", table_name="dim_product")
print(f"load_id={load_id}  environment={environment}  table=dim_product")

from ttfabric.warehouse import gold_target

gold = gold_target(
    spark,
    warehouse="wh_gold",
    schema="dbo",
    write_mode="warehouse_connector",
)


In [ ]:
# ---- Read silver -------------------------------------------------
src = spark.read.table(f"{source_item}.stg_products")
print(f"read {src.count():,} rows from stg_products")


In [ ]:
# ---- Project to the target schema --------------------------------
# Renames come from mappings/gold.yaml `columns`. Applied before the
# business rules, which are written against target names.
src = src.select(
    F.col("product_id"),
    F.col("product_name"),
    F.col("category"),
    F.col("list_price"),
    F.col("stock"),
)


In [ ]:
# Business rule: price_band
# 
src = src.withColumn("price_band", F.expr("""CASE WHEN list_price < 100 THEN 'Budget' WHEN list_price < 500 THEN 'Mid-range' WHEN list_price < 1500 THEN 'Premium' ELSE 'Flagship' END"""))


In [ ]:
# Business rule: stock_status
# 
src = src.withColumn("stock_status", F.expr("""CASE WHEN stock = 0 THEN 'Out of stock' WHEN stock < 50 THEN 'Low' ELSE 'In stock' END"""))


In [ ]:
# ---- SCD type 2 merge --------------------------------------------
# Opens a new version only when a tracked column changes. An edit
# to an untracked column updates in place, so a phone correction
# does not fabricate a history entry.
from ttfabric.dimensions import merge_scd2

merge_scd2(
    spark,
    source=src,
    target_table="dim_product",
    business_key=['product_id'],
    tracked_columns=['list_price', 'category'],
    surrogate_key="product_sk",
    load_id=load_id,
    gold=gold,
)


In [ ]:
# ---- Unknown member ----------------------------------------------
# Guarantees an unmatched fact still joins rather than vanishing
# from a report without trace.
from ttfabric.dimensions import ensure_unknown_member

ensure_unknown_member(
    spark,
    table="dim_product",
    surrogate_key="product_sk",
    key_value=-1,
    defaults={'product_id': 'UNKNOWN', 'product_name': 'Unknown Product', 'category': 'Unknown'},
    gold=gold,
)
dq.flush()
